# Notebook 03: RAG and CAG Pipelines

Builds two competing pipelines over the same article corpus and
runs both against every question in data/qa_dataset.json, at each
cumulative weekly batch size (week1 -> week2 -> week3):

  RAG: embeds articles with bge-m3 into a fresh ChromaDB
        collection per batch, retrieves the top-k passages for
        each question, and asks Qwen3-4B to answer using only
        those passages.
  CAG: skips retrieval entirely: stuffs the FULL batch's article
        text into Qwen3-4B's context window as one big prompt and
        asks it to answer directly. This is where CAG's context
        ceiling becomes visible as the corpus grows — tracked
        explicitly below (articles_included / articles_dropped)
        rather than just failing silently or erroring out.

A question is only answered against a batch if that batch already
contains its source article (earliest_batch <= current batch) —
otherwise you'd be testing the model's own prior knowledge rather
than either pipeline. This gives the same fixed question set
evaluated at three different corpus sizes.

RESUMABLE: results are appended to data/pipeline_results.jsonl as
they're produced; (qa_id, batch, pipeline) triples already in that
file are skipped on re-run. Generation is the slow part here (local
GPU inference, not an API), so this matters just as much as it did
for the Gemini calls in Notebook 02.

Output: data/pipeline_results.json
  One record per (question, batch, pipeline). Notebook 04 uses
  these to compute retrieval-quality metrics (RAG only, hit@k /
  MRR / NDCG@5, same style as the JobAI RAG eval) and LLM-judged
  answer correctness (both pipelines, via Gemini).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/thesis-rag-cag')

Mounted at /content/drive


## Cell 1: Install dependencies

In [2]:
!pip install chromadb sentence-transformers transformers accelerate bitsandbytes -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

## Cell 2: Imports

In [3]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import time
import gc

import torch
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

## Cell 3: Config

In [4]:
DATA_DIR = "data"
WEEK_FILES = ["week1.json", "week2.json", "week3.json"]  # smallest to largest, cumulative
BATCH_NAMES = [f.replace(".json", "") for f in WEEK_FILES]

QA_PATH = os.path.join(DATA_DIR, "qa_dataset.json")
RESULTS_JSONL_PATH = os.path.join(DATA_DIR, "pipeline_results.jsonl")
RESULTS_PATH = os.path.join(DATA_DIR, "pipeline_results.json")

EMBED_MODEL_NAME = "BAAI/bge-m3"          # same embedding model as JobAI's RAG system
GEN_MODEL_NAME = "Qwen/Qwen3-4B"          # plain, non-fine-tuned — same family/size JobAI
                                           # already proved fits on a free Colab GPU in 4-bit

RAG_TOP_K = 5                             # same k as JobAI's RAG system

# Qwen3-4B's native context is 32,768 tokens, but attention memory grows
# quadratically with context length — on a free-tier T4 (shared with the
# loaded model weights), ~5000 tokens is roughly what fits safely without
# OOMing during generation. This cap is itself a real finding: CAG's
# practical ceiling here is set by GPU memory, not the model's nominal
# context window.
MAX_CONTEXT_TOKENS = 5000

GEN_MAX_NEW_TOKENS = 200
GEN_DO_SAMPLE = False  # greedy decoding — deterministic, easier to compare runs

RAG_ANSWER_PROMPT = """Answer the question using ONLY the article excerpts below. If the \
excerpts don't contain the answer, say "Not stated in the provided articles" rather than \
guessing.

Article excerpts:
{context}

Question: {question}

Give a short, direct answer (a phrase or one sentence).
"""

CAG_ANSWER_PROMPT = """You are given a collection of news articles below, each separated by \
"---". Answer the question using ONLY information in these articles. If the articles don't \
contain the answer, say "Not stated in the provided articles" rather than guessing.

Articles:
{context}

Question: {question}

Give a short, direct answer (a phrase or one sentence).
"""

## Cell 4: Load the Q&A set and the weekly article batches

In [5]:
with open(QA_PATH) as f:
    qa_dataset = json.load(f)
print(f"Loaded {len(qa_dataset)} Q&A pairs.")

batches = {}
for filename, batch_name in zip(WEEK_FILES, BATCH_NAMES):
    with open(os.path.join(DATA_DIR, filename)) as f:
        articles = json.load(f)
    batches[batch_name] = {a["id"]: a for a in articles}
    print(f"{batch_name}: {len(articles)} articles")

# Rank batches so we can check "is this question's source article
# already present in this batch" with a simple index comparison.
batch_rank = {name: i for i, name in enumerate(BATCH_NAMES)}


def article_available(earliest_batch, batch_name):
    return batch_rank[earliest_batch] <= batch_rank[batch_name]

Loaded 670 Q&A pairs.
week1: 30 articles
week2: 75 articles
week3: 369 articles


## Cell 5: Load Qwen3-4B in 4-bit

In [6]:
print("Loading Qwen3-4B (4-bit)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
gen_model.eval()

print("Qwen3-4B loaded.")


def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False))


def generate(prompt, max_new_tokens=GEN_MAX_NEW_TOKENS):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(gen_model.device)

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=GEN_DO_SAMPLE,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

Loading Qwen3-4B (4-bit)...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (3 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-4B loaded.


## Cell 6: Load the bge-m3 embedding model

In [7]:
print("Loading bge-m3 embedder...")
embedder = SentenceTransformer(EMBED_MODEL_NAME)
print("bge-m3 loaded.")

Loading bge-m3 embedder...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

... (9 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

bge-m3 loaded.


## Cell 7: RAG pipeline

In [8]:
chroma_client = chromadb.Client()  # in-memory — rebuilt fresh per batch below


def build_rag_index(batch_name, batch_articles):
    """Embeds every article in this batch as a single chunk (articles are
    already short post-NewsAPI-truncation, so no further chunking) and
    loads them into a fresh ChromaDB collection."""
    collection_name = f"rag_{batch_name}"
    try:
        chroma_client.delete_collection(collection_name)  # clean slate on re-run
    except Exception:
        pass
    collection = chroma_client.create_collection(collection_name)

    ids = list(batch_articles.keys())
    documents = [batch_articles[aid]["title"] + "\n\n" + batch_articles[aid]["text"] for aid in ids]
    embeddings = embedder.encode(documents, normalize_embeddings=True).tolist()

    collection.add(ids=ids, documents=documents, embeddings=embeddings)
    return collection


def rag_retrieve(collection, question, k=RAG_TOP_K):
    query_embedding = embedder.encode([question], normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=k)
    retrieved_ids = results["ids"][0]
    retrieved_docs = results["documents"][0]
    retrieved_distances = results["distances"][0]
    return retrieved_ids, retrieved_docs, retrieved_distances


def answer_with_rag(collection, question):
    retrieved_ids, retrieved_docs, retrieved_distances = rag_retrieve(collection, question)
    context = "\n\n---\n\n".join(retrieved_docs)
    prompt = RAG_ANSWER_PROMPT.format(context=context, question=question)
    answer = generate(prompt)
    return {
        "answer": answer,
        "retrieved_article_ids": retrieved_ids,
        "retrieved_distances": retrieved_distances,
    }

## Cell 8: CAG pipeline

In [9]:
def build_cag_context(batch_articles):
    """Concatenates every article's full text, in order, stopping once
    MAX_CONTEXT_TOKENS would be exceeded. Returns the context string plus
    bookkeeping on what got left out — the key signal for how CAG degrades
    as the corpus grows, rather than a silent truncation."""
    included_ids = []
    dropped_ids = []
    pieces = []
    running_tokens = 0

    for aid, article in batch_articles.items():
        piece = f"[{article['title']}]\n{article['text']}"
        piece_tokens = count_tokens(piece)
        if running_tokens + piece_tokens > MAX_CONTEXT_TOKENS:
            dropped_ids.append(aid)
            continue
        pieces.append(piece)
        included_ids.append(aid)
        running_tokens += piece_tokens

    return "\n\n---\n\n".join(pieces), included_ids, dropped_ids, running_tokens


def answer_with_cag(context, question):
    prompt = CAG_ANSWER_PROMPT.format(context=context, question=question)

    # Clean up before the heavy call, not just after — helps avoid
    # fragmentation building up across many calls in a row.
    gc.collect()
    torch.cuda.empty_cache()

    try:
        answer = generate(prompt)
    except torch.cuda.OutOfMemoryError:
        answer = "[OOM Error during CAG generation]"
    finally:
        gc.collect()
        torch.cuda.empty_cache()

    return answer

## Cell 9: Resume support

In [12]:
import gc

def load_progress():
    """Returns a set of (qa_id, batch_name, pipeline) triples already done,
    plus the records themselves, so appended runs never redo work."""
    records = []
    done = set()
    if os.path.exists(RESULTS_JSONL_PATH):
        with open(RESULTS_JSONL_PATH) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                record = json.loads(line)
                records.append(record)
                done.add((record["qa_id"], record["batch"], record["pipeline"]))
    return records, done


results, done_triples = load_progress()
if done_triples:
    print(f"Resuming: {len(done_triples)} (question, batch, pipeline) results already done.")

results_file = open(RESULTS_JSONL_PATH, "a")


def record_result(record):
    results_file.write(json.dumps(record) + "\n")
    results_file.flush()
    os.fsync(results_file.fileno())


# ============================================================
# Cell 10: Main loop — build each batch's indexes once, then run every
# applicable question through both pipelines
# ============================================================

for batch_name in BATCH_NAMES:
    batch_articles = batches[batch_name]
    applicable_qas = [
        qa for qa in qa_dataset if article_available(qa["earliest_batch"], batch_name)
    ]
    print(f"\n=== Batch {batch_name}: {len(batch_articles)} articles, "
          f"{len(applicable_qas)} applicable questions ===")

    # --- RAG index for this batch (built once, reused for every question) ---
    rag_needed = any((qa["qa_id"], batch_name, "rag") not in done_triples for qa in applicable_qas)
    rag_collection = build_rag_index(batch_name, batch_articles) if rag_needed else None

    # --- CAG context for this batch (built once, reused for every question) ---
    cag_needed = any((qa["qa_id"], batch_name, "cag") not in done_triples for qa in applicable_qas)
    if cag_needed:
        cag_context, cag_included_ids, cag_dropped_ids, cag_token_count = build_cag_context(batch_articles)
        print(f"CAG context: {cag_token_count} tokens, "
              f"{len(cag_included_ids)} articles included, {len(cag_dropped_ids)} dropped (over limit)")
        cag_included_set = set(cag_included_ids)
        cag_dropped_set = set(cag_dropped_ids)

    for i, qa in enumerate(applicable_qas):
        qa_id = qa["qa_id"]

        # Aggressively clean up memory to stave off fragmentation
        if i % 10 == 0:
            gc.collect()
            torch.cuda.empty_cache()

        if (qa_id, batch_name, "rag") not in done_triples:
            try:
                rag_result = answer_with_rag(rag_collection, qa["question"])
                record_result({
                    "qa_id": qa_id,
                    "batch": batch_name,
                    "pipeline": "rag",
                    "question": qa["question"],
                    "reference_answer": qa["answer"],
                    "question_type": qa["question_type"],
                    "source_article_id": qa["article_id"],
                    "generated_answer": rag_result["answer"],
                    "retrieved_article_ids": rag_result["retrieved_article_ids"],
                    "retrieved_distances": rag_result["retrieved_distances"],
                    "source_article_retrieved": qa["article_id"] in rag_result["retrieved_article_ids"],
                })
            except Exception as e:
                if "CUDA out of memory" in str(e) or isinstance(e, torch.cuda.OutOfMemoryError):
                    print(f"  [OOM Warning] Out of memory during RAG generation for {qa_id}. Skipping current item.")
                    gc.collect()
                    torch.cuda.empty_cache()
                else:
                    raise e

        if (qa_id, batch_name, "cag") not in done_triples:
            try:
                cag_answer = answer_with_cag(cag_context, qa["question"])
                record_result({
                    "qa_id": qa_id,
                    "batch": batch_name,
                    "pipeline": "cag",
                    "question": qa["question"],
                    "reference_answer": qa["answer"],
                    "question_type": qa["question_type"],
                    "source_article_id": qa["article_id"],
                    "generated_answer": cag_answer,
                    "context_token_count": cag_token_count,
                    "articles_included": len(cag_included_set),
                    "articles_dropped": len(cag_dropped_set),
                    "source_article_included": qa["article_id"] in cag_included_set,
                })
            except Exception as e:
                if "CUDA out of memory" in str(e) or isinstance(e, torch.cuda.OutOfMemoryError):
                    print(f"  [OOM Warning] Out of memory during CAG generation for {qa_id}. Recording fallback.")
                    gc.collect()
                    torch.cuda.empty_cache()
                    record_result({
                        "qa_id": qa_id,
                        "batch": batch_name,
                        "pipeline": "cag",
                        "question": qa["question"],
                        "reference_answer": qa["answer"],
                        "question_type": qa["question_type"],
                        "source_article_id": qa["article_id"],
                        "generated_answer": "[OOM Error during CAG generation]",
                        "context_token_count": cag_token_count,
                        "articles_included": len(cag_included_set),
                        "articles_dropped": len(cag_dropped_set),
                        "source_article_included": qa["article_id"] in cag_included_set,
                    })
                else:
                    raise e

        if (i + 1) % 20 == 0:
            print(f"  {i + 1}/{len(applicable_qas)} questions done for {batch_name}")

    gc.collect()
    torch.cuda.empty_cache()

results_file.close()
print("\nAll batches processed.")

Resuming: 597 (question, batch, pipeline) results already done.

=== Batch week1: 30 articles, 54 applicable questions ===
  20/54 questions done for week1
  40/54 questions done for week1

=== Batch week2: 75 articles, 135 applicable questions ===
  20/135 questions done for week2
  40/135 questions done for week2
  60/135 questions done for week2
  80/135 questions done for week2
  100/135 questions done for week2
  120/135 questions done for week2

=== Batch week3: 369 articles, 670 applicable questions ===
CAG context: 4991 tokens, 45 articles included, 324 dropped (over limit)
  20/670 questions done for week3
  40/670 questions done for week3
  60/670 questions done for week3
  80/670 questions done for week3
  100/670 questions done for week3
  120/670 questions done for week3
  140/670 questions done for week3
  160/670 questions done for week3
  180/670 questions done for week3
  200/670 questions done for week3
  220/670 questions done for week3
  240/670 questions done for w

## Cell 10: Save combined results + quick sanity summary

In [13]:
_, all_records_set = load_progress()  # reload fresh from disk, includes this run's writes
final_records, _ = load_progress()

with open(RESULTS_PATH, "w") as f:
    json.dump(final_records, f, indent=2)
print(f"Saved {len(final_records)} results to {RESULTS_PATH}")

print("\n--- Quick sanity summary ---")
for batch_name in BATCH_NAMES:
    batch_records = [r for r in final_records if r["batch"] == batch_name]
    rag_records = [r for r in batch_records if r["pipeline"] == "rag"]
    cag_records = [r for r in batch_records if r["pipeline"] == "cag"]

    rag_hit = sum(1 for r in rag_records if r.get("source_article_retrieved")) / max(1, len(rag_records))
    cag_included = sum(1 for r in cag_records if r.get("source_article_included")) / max(1, len(cag_records))
    cag_tokens = cag_records[0]["context_token_count"] if cag_records else None

    print(f"{batch_name}: RAG source-article hit rate = {rag_hit:.2%} ({len(rag_records)} questions) | "
          f"CAG context = {cag_tokens} tokens, source-article-included rate = {cag_included:.2%} "
          f"({len(cag_records)} questions)")

print("\nDone. Next step: Notebook 04 — compute retrieval-quality metrics (hit@k / MRR / "
      "NDCG@5) for RAG and run the Gemini LLM-judge answer-correctness check for both "
      "pipelines against data/pipeline_results.json.")

Saved 1718 results to data/pipeline_results.json

--- Quick sanity summary ---
week1: RAG source-article hit rate = 96.30% (54 questions) | CAG context = 3333 tokens, source-article-included rate = 100.00% (54 questions)
week2: RAG source-article hit rate = 97.78% (135 questions) | CAG context = 4991 tokens, source-article-included rate = 59.26% (135 questions)
week3: RAG source-article hit rate = 96.87% (670 questions) | CAG context = 4991 tokens, source-article-included rate = 11.94% (670 questions)

Done. Next step: Notebook 04 — compute retrieval-quality metrics (hit@k / MRR / NDCG@5) for RAG and run the Gemini LLM-judge answer-correctness check for both pipelines against data/pipeline_results.json.
